# ORIE 5530 Project: warm-up extras and station choice

This notebook computes the numbers in the report that `warmup.py` and the Part I and Part II notebooks do not:

1. The extra warm-up numbers (median, rides over 60 minutes, casual rates, splits in Q4 to Q6)
2. The city-wide hourly rides and the share of rides by members
3. The two storm days
4. The candidate stations (report Table 12)
5. The three stations we model (report Table 2) and the distance between W 21 St and Ave A
6. The morning net flow on single days (report Table 13)

Input: `data/trips_clean.parquet` made by clean_data.py. Nothing here is random, so no seed is needed.

The station selection groups rides by station name, because that gives the 2,390 stations of the report, while Part I uses station IDs.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
plt.rcParams["font.family"] = "Inter"
pd.set_option("display.max_columns", None, "display.width", 250)

In [2]:
cols = ["started_at", "ended_at", "start_station_name", "end_station_name", "start_lat", "start_lng",
        "rideable_type", "member_casual", "duration_min"]
trips = pd.read_parquet("data/trips_clean.parquet", columns=cols)
print(f"{len(trips):,} rides")

5,240,216 rides


## 1. Warm-up extras

### Q1. Shape of the distribution (Figure 1 is drawn by `warmup.py`)

In [3]:
durations = trips.duration_min
print(f"median {durations.median():.1f} min, mean {durations.mean():.1f} min")

over_60 = (durations > 60).sum()
print(f"rides over 60 minutes, left out of the plot: {over_60:,} ({over_60 / len(trips):.1%})")

median 9.8 min, mean 13.5 min
rides over 60 minutes, left out of the plot: 61,012 (1.2%)


In [4]:
# 1 minute bins up to 60 minutes
bins = np.arange(0, 61, 1)
counts, _ = np.histogram(durations[durations <= 60], bins=bins)

peak = counts.argmax()
print(f"tallest bin: {bins[peak]} to {bins[peak + 1]} minutes")

tallest bin: 5 to 6 minutes


### Q2 and Q3. Standard deviation and the casual rate over 20 minutes

In [5]:
print(f"standard deviation {durations.std(ddof=0):.2f} min")

is_casual = trips.member_casual.eq("casual")
casual_over_20 = (is_casual & durations.gt(20)).sum()
print(f"P(duration > 20 | casual) = {casual_over_20:,} / {is_casual.sum():,} = {casual_over_20 / is_casual.sum():.3f}")

standard deviation 12.76 min
P(duration > 20 | casual) = 356,628 / 1,123,811 = 0.317


### Q4. Pieces of Bayes' rule and the shares of members and casual users

In [6]:
is_member = trips.member_casual.eq("member")
over_25 = durations.gt(25)

p_over_25_given_member = (over_25 & is_member).sum() / is_member.sum()
p_over_25_given_casual = (over_25 & is_casual).sum() / is_casual.sum()
p_member = is_member.mean()
p_over_25 = over_25.mean()

print(f"P(duration > 25 | member) = {p_over_25_given_member:.5f}")
print(f"P(member) = {p_member:.5f}")
print(f"P(duration > 25) = {p_over_25:.5f}")
print(f"Bayes: {p_over_25_given_member * p_member / p_over_25:.3f}")
print(f"P(duration > 25 | casual) = {p_over_25_given_casual:.3f}")

P(duration > 25 | member) = 0.09345
P(member) = 0.78554
P(duration > 25) = 0.12192
Bayes: 0.602
P(duration > 25 | casual) = 0.226


In [7]:
share_all = trips.member_casual.value_counts(normalize=True)
share_long = trips.member_casual[over_25].value_counts(normalize=True)
pd.DataFrame({"all rides": share_all, "rides over 25 min": share_long}).round(3)

,all rides,rides over 25 min
member_casual,,
member,0.786,0.602
casual,0.214,0.398


### Q5. Average duration by bike type within each user type

In [8]:
by_type = trips.groupby(["member_casual", "rideable_type"]).duration_min.mean().unstack()
by_type.round(1)

rideable_type,classic_bike,electric_bike
member_casual,,
casual,21.6,18.0
member,11.6,12.2


### Q6. Share of electric bikes, overall and for short rides

In [9]:
is_electric = trips.rideable_type.eq("electric_bike")
under_5 = durations.lt(5)
from_5_to_10 = durations.ge(5) & durations.lt(10)

print(f"all rides:           {is_electric.mean():.1%}")
print(f"under 10 minutes:    {is_electric[durations.lt(10)].mean():.1%}")
print(f"under 5 minutes:     {is_electric[under_5].mean():.1%}")
print(f"5 to 10 minutes:     {is_electric[from_5_to_10].mean():.1%}")

all rides:           73.1%
under 10 minutes:    72.0%
under 5 minutes:     67.9%
5 to 10 minutes:     74.8%


## 2. Rides across the city by hour (report Part I, "Stations and time blocks")

A ride counts in the hour it starts. We average over the 21 weekdays and the 10 weekend days of August.

In [10]:
start = trips.started_at
is_august = start.dt.month == 8
is_weekday = start.dt.dayofweek < 5

weekday_rides = trips[is_august & is_weekday]
weekend_rides = trips[is_august & ~is_weekday]
n_weekdays = weekday_rides.started_at.dt.date.nunique()
n_weekend_days = weekend_rides.started_at.dt.date.nunique()
print(n_weekdays, "weekdays,", n_weekend_days, "weekend days")

21 weekdays, 10 weekend days


In [11]:
weekday_per_hour = weekday_rides.started_at.dt.hour.value_counts().sort_index() / n_weekdays
weekend_per_hour = weekend_rides.started_at.dt.hour.value_counts().sort_index() / n_weekend_days

per_hour = pd.DataFrame({"weekday": weekday_per_hour, "weekend day": weekend_per_hour}).round(0)
per_hour.index.name = "hour"
per_hour

,weekday,weekend day
hour,,
0,2070.0,4566.0
1,1145.0,3329.0
2,638.0,2267.0
3,404.0,1487.0
4,479.0,1000.0
5,1362.0,754.0
6,3841.0,1340.0
7,7077.0,2478.0
8,11936.0,4377.0


In [12]:
# the peaks and the early afternoon on weekdays
print(f"8am: {weekday_per_hour[8]:,.0f} rides per hour")
print(f"5pm: {weekday_per_hour[17]:,.0f} rides per hour")
print(f"12pm to 3pm: {weekday_per_hour[[12, 13, 14]].round(0).tolist()}")

8am: 11,936 rides per hour
5pm: 16,317 rides per hour
12pm to 3pm: [7780.0, 8155.0, 8511.0]


In [13]:
in_morning = weekday_rides.started_at.dt.hour.between(7, 9)
member_share_morning = weekday_rides.member_casual[in_morning].eq("member").mean()
member_share_weekend = weekend_rides.member_casual.eq("member").mean()

print(f"members, weekday rides 7 to 10am: {member_share_morning:.0%}")
print(f"members, all weekend rides:       {member_share_weekend:.0%}")

members, weekday rides 7 to 10am: 89%
members, all weekend rides:       71%


## 3. The two storm days

Rides in one hour of the day against the average of the same hour over the 21 weekdays.

In [14]:
rides_by_day_hour = weekday_rides.groupby([weekday_rides.started_at.dt.date, weekday_rides.started_at.dt.hour]).size()
rides_by_day_hour = rides_by_day_hour.unstack(fill_value=0)

for day, hour in [("2026-08-20", 18), ("2026-08-27", 16)]:
    rides = rides_by_day_hour.loc[pd.Timestamp(day).date(), hour]
    normal = rides_by_day_hour[hour].mean()
    print(f"{day} at {hour}:00: {rides:,} rides against {normal:,.0f} on average, {rides / normal:.0%}")

2026-08-20 at 18:00: 1,051 rides against 14,785 on average, 7%
2026-08-27 at 16:00: 1,988 rides against 11,741 on average, 17%


## 4. Candidate stations (report Table 12)

Here we group the rides by station name, as when we chose the stations. A ride is a departure (-1) at its start
station and an arrival (+1) at its end station, counted at its own time.

In [15]:
departures = pd.DataFrame({"station": trips.start_station_name, "time": trips.started_at, "event": -1,
                           "user": trips.member_casual})
arrivals = pd.DataFrame({"station": trips.end_station_name, "time": trips.ended_at, "event": 1,
                         "user": trips.member_casual})
events = pd.concat([departures, arrivals], ignore_index=True).dropna(subset=["station"])
events = events[events.time.dt.month == 8]

on_weekday = events.time.dt.dayofweek < 5
weekday_events = events[on_weekday]
weekend_events = events[~on_weekday]
print(weekday_events.station.nunique(), "stations")

2390 stations


In [16]:
trips_per_weekday = weekday_events.groupby("station").size() / n_weekdays
rank = trips_per_weekday.rank(ascending=False, method="first").astype(int)

# net flow = arrivals minus departures, averaged over the 21 weekdays
hour = weekday_events.time.dt.hour
net_morning = weekday_events[hour.between(6, 11)].groupby("station").event.sum() / n_weekdays
net_evening = weekday_events[hour.between(15, 20)].groupby("station").event.sum() / n_weekdays

casual_share = weekday_events.user.eq("casual").groupby(weekday_events.station).mean()
trips_per_weekend_day = weekend_events.groupby("station").size() / n_weekend_days

In [17]:
CANDIDATES = {
    "Office": ["W 21 St & 6 Ave", "E 17 St & Broadway", "Park Ave & E 42 St"],
    "Residential": ["11 Ave & W 41 St", "Metropolitan Ave & Bedford Ave", "Ave A & E 14 St",
                    "Clinton St & Grand St", "Columbus Ave & W 72 St"],
    "Park": ["Pier 61 at Chelsea Piers", "7 Ave & Central Park South", "Central Park S & 6 Ave",
             "5 Ave & E 72 St"],
}

rows = []
for kind, stations in CANDIDATES.items():
    for s in stations:
        rows.append({
            "kind": kind,
            "station": s,
            "rank": rank[s],
            "trips per weekday": round(trips_per_weekday[s]),
            "net 6am-12pm": round(net_morning.get(s, 0), 1),
            "net 3-9pm": round(net_evening.get(s, 0), 1),
            "casual": f"{casual_share[s]:.0%}",
            "weekend / weekday": round(trips_per_weekend_day[s] / trips_per_weekday[s], 2),
        })
pd.DataFrame(rows)

,kind,station,rank,trips per weekday,net 6am-12pm,net 3-9pm,casual,weekend / weekday
0,Office,W 21 St & 6 Ave,1,1192,53.6,-49.4,11%,0.88
1,Office,E 17 St & Broadway,5,1011,63.1,-52.5,13%,1.08
2,Office,Park Ave & E 42 St,9,913,58.1,-42.6,15%,0.66
3,Residential,11 Ave & W 41 St,11,889,-46.4,29.6,16%,1.07
4,Residential,Metropolitan Ave & Bedford Ave,16,828,7.8,5.1,16%,1.14
5,Residential,Ave A & E 14 St,38,725,-25.2,21.4,11%,1.13
6,Residential,Clinton St & Grand St,45,691,-30.6,33.2,11%,1.10
7,Residential,Columbus Ave & W 72 St,50,651,-42.8,37.6,15%,1.14
8,Park,Pier 61 at Chelsea Piers,2,1092,4.7,-23.7,22%,0.96
9,Park,7 Ave & Central Park South,13,870,19.6,-31.5,48%,1.39


## 5. The three stations we model (report Table 2)

Day swing: we add up the average net flow hour by hour from midnight and take the highest minus the lowest point.
It is how much the number of bikes would rise and fall over an average weekday if staff moved no bikes.

In [18]:
# docks from Citi Bike's live station feed (station_information.json), read in November 2026
CHOSEN = {"W 21 St & 6 Ave": 74, "Ave A & E 14 St": 41, "7 Ave & Central Park South": 81}

def day_swing(station):
    e = weekday_events[weekday_events.station == station]
    net_per_hour = e.groupby(e.time.dt.hour).event.sum().reindex(range(24), fill_value=0) / n_weekdays
    bikes = np.concatenate([[0], net_per_hour.cumsum().values])
    return bikes.max() - bikes.min()

In [19]:
rows = []
for s, docks in CHOSEN.items():
    rows.append({
        "station": s,
        "docks": docks,
        "net 6am-12pm": round(net_morning[s], 1),
        "net 3-9pm": round(net_evening[s], 1),
        "day swing": round(day_swing(s)),
    })
pd.DataFrame(rows)

,station,docks,net 6am-12pm,net 3-9pm,day swing
0,W 21 St & 6 Ave,74,53.6,-49.4,59
1,Ave A & E 14 St,41,-25.2,21.4,26
2,7 Ave & Central Park South,81,19.6,-31.5,38


In [20]:
# distance between W 21 St and Ave A, from the median start coordinates of each station
coords = trips[trips.start_station_name.isin(["W 21 St & 6 Ave", "Ave A & E 14 St"])]
coords = coords.groupby("start_station_name")[["start_lat", "start_lng"]].median()
lat1, lng1 = np.radians(coords.loc["W 21 St & 6 Ave"])
lat2, lng2 = np.radians(coords.loc["Ave A & E 14 St"])

# haversine formula, Earth radius 6371 km
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
print(f"W 21 St to Ave A: {2 * 6371 * np.arcsin(np.sqrt(a)):.1f} km")

W 21 St to Ave A: 1.7 km


## 6. Morning net flow on single days (report Table 13)

Net flow from 6am to 12pm on each of the 21 weekdays. "Same direction" counts the days with the same sign as the mean.

In [21]:
morning_events = weekday_events[weekday_events.time.dt.hour.between(6, 11)]
morning_events = morning_events.assign(day=morning_events.time.dt.date)
weekday_dates = sorted(morning_events.day.unique())

SHOWN = ["W 21 St & 6 Ave", "E 17 St & Broadway", "Ave A & E 14 St", "Clinton St & Grand St",
         "Columbus Ave & W 72 St", "7 Ave & Central Park South", "Central Park S & 6 Ave"]
DAYS = ["2026-08-04", "2026-08-12", "2026-08-20"]

In [22]:
rows = []
for s in SHOWN:
    e = morning_events[morning_events.station == s]
    net_per_day = e.groupby("day").event.sum().reindex(weekday_dates, fill_value=0)
    row = {"station": s}
    for d in DAYS:
        row[d] = net_per_day[pd.Timestamp(d).date()]
    row["mean"] = round(net_per_day.mean(), 1)
    row["sd"] = round(net_per_day.std(), 1)
    row["same direction"] = (np.sign(net_per_day) == np.sign(net_per_day.mean())).sum()
    rows.append(row)
pd.DataFrame(rows)

,station,2026-08-04,2026-08-12,2026-08-20,mean,sd,same direction
0,W 21 St & 6 Ave,62,45,56,53.6,9.3,21
1,E 17 St & Broadway,69,72,70,63.1,13.0,21
2,Ave A & E 14 St,-32,-33,-29,-25.2,12.0,21
3,Clinton St & Grand St,-46,-45,-20,-30.6,10.0,21
4,Columbus Ave & W 72 St,-76,-44,-35,-42.8,15.0,21
5,7 Ave & Central Park South,55,50,-9,19.6,22.9,16
6,Central Park S & 6 Ave,53,69,61,39.6,23.9,21
